In [ ]:
from preprocesado import seleccion_caracteristicas

from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier   # Asegurarse de tener xgboost instalado: pip install xgboost
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

ModuleNotFoundError: No module named 'script2'

#### Comparador de num caracteristicas XGB

In [ ]:
def comparar_xgb(X_train_linear, y_train_linear, test_linear_pre, tscv, scorer):
    mejor_score_xgb = -1
    mejor_k_xgb = None
    mejor_modelo_xgb = None
    mejor_selector_xgb = None

    for k in [30, 35, 40]:
        # ── 1. Selección de características ──────────────────────────────────
        X_train_xgb_red, selector_xgb, _ = seleccion_caracteristicas(
            X_train_linear, y_train_linear,
            metodo='kbest',
            k=k
        )

        # ── 2. RandomizedSearchCV ─────────────────────────────────────────────
        search_xgb = RandomizedSearchCV(
            XGBClassifier(
                objective="multi:softmax", num_class=4,
                random_state=42, n_jobs=-1
            ),
            param_distributions={
                'n_estimators':     [200, 300, 500],
                'max_depth':        [4, 5, 6, 7, 8],
                'learning_rate':    [0.01, 0.05, 0.1, 0.2],
                'subsample':        [0.6, 0.7, 0.8, 0.9],
                'colsample_bytree': [0.6, 0.7, 0.8, 0.9],
                'min_child_weight': [1, 3, 5],
                'gamma':            [0, 0.1, 0.3, 0.5],
                'reg_alpha':        [0, 0.1, 0.5],
                'reg_lambda':       [1, 1.5, 2],
            },
            n_iter=50,
            cv=tscv,
            scoring=scorer,
            random_state=42,
            n_jobs=-1,
            verbose=0
        )
        search_xgb.fit(X_train_xgb_red, y_train_linear)

        print(f"k={k:2d} | F1: {search_xgb.best_score_:.4f} | params: {search_xgb.best_params_}")

        # ── 3. Guardar si es el mejor ─────────────────────────────────────────
        if search_xgb.best_score_ > mejor_score_xgb:
            mejor_score_xgb    = search_xgb.best_score_
            mejor_k_xgb        = k
            mejor_modelo_xgb   = search_xgb.best_estimator_
            mejor_selector_xgb = selector_xgb

    print(f"\n✅ Mejor k XGB: {mejor_k_xgb} | Mejor F1: {mejor_score_xgb:.4f}")

    # ── Aplicar mejor selector al test ───────────────────────────────────────────
    cols_xgb = X_train_linear.columns[mejor_selector_xgb.get_support()]
    X_test_xgb_red = test_linear_pre[cols_xgb]

#### Comparador de n caracteristicas MLP


In [ ]:
def comparar_mlp(X_train_linear, y_train_linear, test_linear_pre, tscv, scorer):
    mejor_score_mlp = -1
    mejor_k_mlp = None
    mejor_modelo_mlp = None
    mejor_selector_mlp = None

    for k in [30, 35, 40]:
        # ── 1. Selección de características ──────────────────────────────────
        X_train_mlp_red, selector_mlp, _ = seleccion_caracteristicas(
            X_train_linear, y_train_linear,
            metodo='kbest',
            k=k
        )

        # ── 2. RandomizedSearchCV ─────────────────────────────────────────────
        search_mlp = RandomizedSearchCV(
            MLPClassifier(solver='adam', random_state=42),
            param_distributions={
                'hidden_layer_sizes': [(64,), (128,), (64, 32), (128, 64),
                                    (256, 128), (128, 64, 32), (256, 128, 64)],
                'alpha':              [0.001, 0.01, 0.1],
                'learning_rate_init': [0.001],
                'max_iter':           [200, 300, 500],
                'early_stopping':     [True],
                'validation_fraction':[0.1],
                'batch_size':         [8, 16, 32],
                'activation':         ['relu', 'tanh'],
            },
            n_iter=50,
            cv=tscv,
            scoring=scorer,
            random_state=42,
            n_jobs=-1,
            verbose=0  # silenciamos para no saturar output en el loop
        )
        search_mlp.fit(X_train_mlp_red, y_train_linear)

        print(f"k={k:2d} | F1: {search_mlp.best_score_:.4f} | params: {search_mlp.best_params_}")

        # ── 3. Guardar si es el mejor ─────────────────────────────────────────
        if search_mlp.best_score_ > mejor_score_mlp:
            mejor_score_mlp   = search_mlp.best_score_
            mejor_k_mlp       = k
            mejor_modelo_mlp  = search_mlp.best_estimator_
            mejor_selector_mlp = selector_mlp

    print(f"\n✅ Mejor k MLP: {mejor_k_mlp} | Mejor F1: {mejor_score_mlp:.4f}")

    # ── Aplicar mejor selector al test ───────────────────────────────────────────
    cols_mlp = X_train_linear.columns[mejor_selector_mlp.get_support()]
    X_test_mlp_red = test_linear_pre[cols_mlp]

#### Comparador de n caracteristicas KNN

In [ ]:
def comparar_knn(X_train_linear, y_train_linear, test_linear_pre, tscv, scorer):
    mejor_score_knn = -1
    mejor_k_knn = None
    mejor_modelo_knn = None
    mejor_selector_knn = None

    for k in [30, 35, 40]:
        # ── 1. Selección de características ──────────────────────────────────────
        X_train_knn_red, selector_knn, scores_knn = seleccion_caracteristicas(
            X_train_linear, y_train_linear,
            metodo='kbest',
            k=k
        )
        X_test_knn_red = test_linear_pre[X_train_linear.columns[selector_knn.get_support()]]

        # ── 2. Grid de hiperparámetros ────────────────────────────────────────────
        param_grid_knn = {
            'n_neighbors':  [3, 5, 7, 9, 11, 15, 21],
            'weights':      ['uniform', 'distance'],
            'metric':       ['euclidean', 'manhattan', 'minkowski'],
            'p':            [1, 2],
            'leaf_size':    [20, 30, 40],
        }

        # ── 3. RandomizedSearchCV ─────────────────────────────────────────────────
        search_knn = RandomizedSearchCV(
            KNeighborsClassifier(),
            param_distributions=param_grid_knn,
            n_iter=50,
            cv=tscv,
            scoring=scorer,
            random_state=42,
            n_jobs=-1,
            verbose=1
        )
        search_knn = entrenar_o_cargar(search_knn, X_train_knn_red, y_train_linear, f"models/search_knn_k{k}.pkl")

        print(f"k={k} | F1: {search_knn.best_score_:.4f} | params: {search_knn.best_params_}")

        # ── 4. Guardar si es el mejor hasta ahora ─────────────────────────────────
        if search_knn.best_score_ > mejor_score_knn:
            mejor_score_knn  = search_knn.best_score_
            mejor_k_knn      = k
            mejor_modelo_knn = search_knn.best_estimator_
            mejor_selector_knn = selector_knn

    print(f"\nMejor k global: {mejor_k_knn} | Mejor F1: {mejor_score_knn:.4f}")

    # ── 5. Aplicar el mejor selector al test ─────────────────────────────────────
    X_test_knn_final = test_linear_pre[X_train_linear.columns[mejor_selector_knn.get_support()]]

#### Comparador num características LogReg

In [ ]:
def comparar_logres(X_train_linear, y_train_linear, test_linear_pre, tscv, scorer):
    mejor_score_logres = -1
    mejor_k_logres = None
    mejor_modelo_logres = None
    mejor_selector_logres = None

    for k in [30, 35, 40]:
        # ── 1. Selección de características ──────────────────────────────────
        X_train_logres_red, selector_logres, _ = seleccion_caracteristicas(
            X_train_linear, y_train_linear,
            metodo='kbest',
            k=k
        )

        # ── 2. RandomizedSearchCV ─────────────────────────────────────────────
        search_logres = RandomizedSearchCV(
            LogisticRegression(multi_class='multinomial', random_state=42),
            param_distributions={
                'C':            [0.001, 0.01, 0.1, 1, 10, 100],
                'penalty':      ['l1', 'l2', 'elasticnet'],
                'l1_ratio':     [0.1, 0.3, 0.5, 0.7, 0.9],
                'solver':       ['saga'],
                'class_weight': ['balanced', None],
                'max_iter':     [500, 1000],
            },
            n_iter=75,
            cv=tscv,
            scoring=scorer,
            random_state=42,
            n_jobs=-1,
            verbose=0
        )
        search_logres.fit(X_train_logres_red, y_train_linear)

        print(f"k={k:2d} | F1: {search_logres.best_score_:.4f} | params: {search_logres.best_params_}")

        # ── 3. Guardar si es el mejor ─────────────────────────────────────────
        if search_logres.best_score_ > mejor_score_logres:
            mejor_score_logres    = search_logres.best_score_
            mejor_k_logres        = k
            mejor_modelo_logres   = search_logres.best_estimator_
            mejor_selector_logres = selector_logres

    print(f"\n✅ Mejor k LogRes: {mejor_k_logres} | Mejor F1: {mejor_score_logres:.4f}")

    # ── Aplicar mejor selector al test ───────────────────────────────────────────
    cols_logres = X_train_linear.columns[mejor_selector_logres.get_support()]
    X_test_logres_red = test_linear_pre[cols_logres]